# Data Loading, Cleaning & Inspection

This notebook loads a real CSV, inspects the raw structure, cleans the dataset, and exports a validated cleaned version to CSV and Parquet.

The code is written to run even if the network source is unavailable by falling back to a small embedded CSV sample.

In [1]:
from io import StringIO
from pathlib import Path

import pandas as pd

# Resolve the repository root so the notebook works from both VS Code and Jupyter.
repo_root = Path.cwd() if (Path.cwd() / "README.md").exists() else Path.cwd().parent
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)

source_url = "https://raw.githubusercontent.com/selva86/datasets/master/Life_Expectancy_Data.csv"
source_path = data_dir / "india_life_expectancy.csv"

# Tiny offline fallback so the notebook stays runnable without network access.
fallback_csv = """Country,Year,Status,Life expectancy ,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles , BMI ,under-five deaths ,Polio,Total expenditure,Diphtheria , HIV/AIDS,GDP,Population, thinness  1-19 years, thinness 5-9 years,Income composition of resources,Schooling
India,2014,Developing,68.0,184,957,3.07,86.52153895,79,79563,18.1,1200,84,4.69,85,0.2,1573.11889,1.293859294e9,26.8,27.4,0.607,11.6
India,2013,Developing,67.6,187,1000,3.11,67.67230438,7,13822,17.5,1300,82,4.53,83,0.2,1452.195373,1.27856227e9,26.8,27.5,0.599,11.5
India,2012,Developing,67.3,19,1100,3.1,64.96964491,73,18668,17.0,1400,79,4.39,82,0.2,1446.98541,1.26365852e9,26.9,27.6,0.59,11.3
India,2011,Developing,66.8,193,1100,3.0,64.6059005,44,33634,16.4,1500,79,4.33,82,0.2,1461.671957,1.24723629e8,26.9,27.7,0.58,10.8
India,2010,Developing,66.4,196,1200,2.77,57.73359864,38,31458,15.9,1600,76,4.28,79,0.2,1345.77153,1.2398691e7,27.0,27.8,0.569,10.4
"""

# Load the India slice from a cached CSV if available, otherwise fetch and cache it.
try:
    if source_path.exists():
        raw_df = pd.read_csv(source_path)
    else:
        world_df = pd.read_csv(source_url)
        raw_df = world_df.loc[world_df["Country"].eq("India")].copy()
        raw_df.to_csv(source_path, index=False)
except Exception:
    raw_df = pd.read_csv(StringIO(fallback_csv))
    raw_df.to_csv(source_path, index=False)

# Inspect the raw dataset before cleaning.
india_df = pd.read_csv(source_path)
print("Shape:", india_df.shape)
print("\nDtypes:")
print(india_df.dtypes)
print("\nHead(10):")
print(india_df.head(10).to_string(index=False))

inspection_report = pd.DataFrame(
    {
        "missing_values": india_df.isna().sum(),
        "non_null": india_df.notna().sum(),
        "duplicates": [india_df.duplicated().sum()] * len(india_df.columns),
    },
    index=india_df.columns,
)
print("\nInspection summary:")
print(inspection_report.to_string())

# Clean the data for analysis and export.
clean_df = india_df.rename(columns=lambda column: column.strip().lower().replace(" ", "_"))
clean_df["country"] = clean_df["country"].astype(str).str.strip()
clean_df["status"] = clean_df["status"].astype(str).str.strip().str.title()

numeric_columns = clean_df.columns.difference(["country", "status"])
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

clean_df = clean_df.drop_duplicates().sort_values(["year", "country"], ascending=[False, True]).reset_index(drop=True)

for column in clean_df.select_dtypes(include="number").columns:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

print("\nCleaned sample:")
print(clean_df.head(10).to_string(index=False))

# Lightweight validation checks.
assert not clean_df.empty, "Cleaned DataFrame should not be empty"
assert clean_df["country"].eq("India").all(), "The dataset should contain India rows only"
assert clean_df.duplicated().sum() == 0, "Duplicate rows should be removed"
assert clean_df.columns.is_unique, "Columns should remain unique after cleaning"

cleaned_path = data_dir / "w1d3_cleaned_india_life_expectancy.csv"
parquet_path = data_dir / "w1d3_cleaned_india_life_expectancy.parquet"
clean_df.to_csv(cleaned_path, index=False)
clean_df.to_parquet(parquet_path, index=False)

size_report = pd.DataFrame(
    {
        "format": ["CSV", "Parquet"],
        "path": [cleaned_path.name, parquet_path.name],
        "size_bytes": [cleaned_path.stat().st_size, parquet_path.stat().st_size],
    }
)
size_report["size_kb"] = (size_report["size_bytes"] / 1024).round(2)
print("\nExport size comparison:")
print(size_report.to_string(index=False))

smaller = size_report.sort_values("size_bytes").iloc[0]
larger = size_report.sort_values("size_bytes").iloc[-1]
print(f"\n{smaller['format']} is smaller than {larger['format']} by {larger['size_bytes'] - smaller['size_bytes']:,} bytes.")

assert cleaned_path.exists(), "CSV export should exist"
assert parquet_path.exists(), "Parquet export should exist"
assert cleaned_path.stat().st_size > 0, "CSV export should not be empty"
assert parquet_path.stat().st_size > 0, "Parquet export should not be empty"

Shape: (11, 22)

Dtypes:
Country                                str
Year                                 int64
Status                                 str
Life expectancy                    float64
Adult Mortality                      int64
infant deaths                        int64
Alcohol                            float64
percentage expenditure             float64
Hepatitis B                          int64
Measles                              int64
 BMI                               float64
under-five deaths                    int64
Polio                                int64
Total expenditure                  float64
Diphtheria                           int64
 HIV/AIDS                          float64
GDP                                float64
Population                         float64
 thinness  1-19 years              float64
 thinness 5-9 years                float64
Income composition of resources    float64
Schooling                          float64
dtype: object

Head(10):
Coun

## Notes

The cleaned India slice is written to `data/w1d3_cleaned_india_life_expectancy.csv` and `data/w1d3_cleaned_india_life_expectancy.parquet`.
The notebook also runs validation assertions so the cleaned output stays reproducible and analysis-ready.